In [11]:
import pandas as pd

raw = pd.read_excel('BuildFest_EveryMind_Survey_Data_FULL.xlsx')
df = raw.copy()
df

,Id,Start time,Completion time,Language,Are you a....?,Please enter the date of your training.,Which county is this training affiliated with?,Please tell us how much you disagree or agree with the statements..I can identify mental health challenges and crises.,Please tell us how much you disagree or agree with the statements..I can describe 988 and how it works.,Please tell us how much you disagree or agree with the statements..I know when to use 988.,Now think back to before you took part in the Understanding 988 training. .I could identify mental health challenges and crises.,Now think back to before you took part in the Understanding 988 training. .I could describe 988 and how it works.,Now think back to before you took part in the Understanding 988 training. .I knew when to use 988.,How much do you agree or disagree with the following aspects of the training?.The presenter(s) engaged the participants.,How much do you agree or disagree with the following aspects of the training?.The presenter(s) knew the subject matter well.,How much do you agree or disagree with the following aspects of the training?.The training was well organized.,How likely are you to recommend the Understanding 988 training to a friend or colleague?,How likely are you to call 988 if you or someone else is experiencing a mental health challenge or crisis?,What was the most valuable thing you learned during the training?,How can we improve this training in the future?
0,1,2025-10-21 14:08:09,2025-10-21 14:09:59,English (United States)‎,School staff member;,2025-10-21,Anne Arundel,Strongly agree,Strongly agree,Strongly agree,Agree,Agree,Strongly agree,Agree,Strongly agree,Strongly agree,8.0,NaN,How the organization is handling cut funding t...,NaN
1,2,2025-10-21 14:08:07,2025-10-21 14:10:06,English (United States)‎,School staff member;,2025-10-21,Anne Arundel,Strongly agree,Agree,Strongly agree,Agree,Disagree,Disagree,Strongly agree,Strongly agree,Strongly agree,9.0,NaN,The understanding and purpose for platform,NaN
2,3,2025-10-21 14:08:08,2025-10-21 14:10:22,English (United States)‎,School staff member;,2025-10-21,Anne Arundel,Strongly agree,Strongly agree,Strongly agree,Agree,Agree,Agree,Strongly agree,Strongly agree,Strongly agree,10.0,NaN,That you don’t have to be in crisis to use 988,It was awesome!
3,4,2025-10-21 14:08:00,2025-10-21 14:10:30,English (United States)‎,School staff member;,2025-10-21,Anne Arundel,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Agree,Agree,Agree,Strongly agree,Strongly agree,10.0,NaN,I like the pre and post test \nI also like tha...,NaN
4,5,2025-10-21 14:09:38,2025-10-21 14:10:49,English (United States)‎,School staff member;,2025-10-21,Anne Arundel,Strongly agree,Agree,Strongly agree,Strongly agree,Neither agree nor disagree,Agree,Strongly agree,Strongly agree,Strongly agree,9.0,NaN,The range of services offered through 988,Allow for a longer training with more interact...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,1295,2026-10-25 00:40:44,2026-10-25 00:41:10,English (United States)‎,Student,2026-10-09,Anne Arundel,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,Strongly agree,10.0,NaN,988 is for everyone,NaN
1295,1296,2026-10-25 00:41:14,2026-10-25 00:41:36,English (United States)‎,Student,2026-10-09,Anne Arundel,Neither agree nor disagree,Agree,Agree,Neither agree nor disagree,Disagree,Neither agree nor disagree,NaN,NaN,NaN,5.0,NaN,NaN,NaN
1296,1297,2026-10-25 00:41:38,2026-10-25 00:42:14,English (United States)‎,Student,2026-10-09,Anne Arundel,Strongly disagree,Strongly disagree,Strongly disagree,Strongly disagree,Strongly disagree,Strongly disagree,Disagree,Disagree,Strongly disagree,5.0,NaN,You don't have to call for suicide thoughts,NaN
1297,1298,2026-10-25 00:42:17,2026-10-25 00:42:45,English (United States)‎,Student,2026-10-09,Anne Arundel,Strongly agree,Strongly agree,Strongly agree,

In [ ]:
#FOR MORE INFORMATION ABOUT DATASET AND COLUMNS, SEE DATA DICTIONARY

#Renaming Data Columns to be more understandable

df.columns = [
    "response_id", "survey_start", "survey_end", "survey_language", "role",
    "training_date_reported", "county",
    "post_identify_crisis", "post_describe_988", "post_know_when_988",
    "pre_identify_crisis", "pre_describe_988", "pre_know_when_988",
    "presenter_engaged", "presenter_knowledgeable", "training_organized",
    "recommend_score_0_10", "likely_to_call_988_0_10",
    "text_most_valuable", "text_how_to_improve",
]

# clean hidden characters in all text columns
for c in df.select_dtypes(["object", "string"]):
    df[c] = (df[c].astype("string")
                  .str.replace("\u200e", "", regex=False)
                  .str.replace("\u00a0", " ", regex=False)
                  .str.strip())

# ratings: text -> 1-5
scale = {"strongly disagree": 1, "disagree": 2, "neither agree nor disagree": 3,
         "neither agree or disagree": 3, "agree": 4, "strongly agree": 5}
rating_cols = [c for c in df.columns if c.startswith(("pre_", "post_", "presenter_", "training_org"))]
for c in rating_cols:
    df[c] = df[c].str.lower().map(scale)

# counties: 15 spellings -> 6
df["county"] = (df.county.str.lower()
                  .str.replace(" county", "", regex=False)
                  .str.replace(" with nami mc", "", regex=False)
                  .str.title()
                  .replace({"Queen Anne'S": "Queen Anne's", "Prince George'S": "Prince George's"}))

# the 6 columns you create
df["survey_start"] = pd.to_datetime(df.survey_start)
df["survey_end"]   = pd.to_datetime(df.survey_end)
df["duration_sec"] = (df.survey_end - df.survey_start).dt.total_seconds()
df["is_student"]   = df.role.str.contains("Student", na=False)
df["gain_identify_crisis"] = df.post_identify_crisis - df.pre_identify_crisis
df["gain_describe_988"]    = df.post_describe_988    - df.pre_describe_988
df["gain_know_when_988"]   = df.post_know_when_988   - df.pre_know_when_988
six = [c for c in df.columns if c.startswith(("pre_", "post_")) and "988" in c or "crisis" in c and c.startswith(("pre_","post_"))]
df["exclude_candidate"] = (df[six] == 1).all(axis=1)


df


,response_id,survey_start,survey_end,survey_language,role,training_date_reported,county,post_identify_crisis,post_describe_988,post_know_when_988,...,recommend_score_0_10,likely_to_call_988_0_10,text_most_valuable,text_how_to_improve,duration_sec,is_student,gain_identify_crisis,gain_describe_988,gain_know_when_988,exclude_candidate
0,1,2025-10-21 14:08:09,2025-10-21 14:09:59,English (United States),School staff member;,2025-10-21,Anne Arundel,5,5,5,...,8.0,NaN,How the organization is handling cut funding t...,<NA>,110.0,False,1,1,0,False
1,2,2025-10-21 14:08:07,2025-10-21 14:10:06,English (United States),School staff member;,2025-10-21,Anne Arundel,5,4,5,...,9.0,NaN,The understanding and purpose for platform,<NA>,119.0,False,1,2,3,False
2,3,2025-10-21 14:08:08,2025-10-21 14:10:22,English (United States),School staff member;,2025-10-21,Anne Arundel,5,5,5,...,10.0,NaN,That you don’t have to be in crisis to use 988,It was awesome!,134.0,False,1,1,1,False
3,4,2025-10-21 14:08:00,2025-10-21 14:10:30,English (United States),School staff member;,2025-10-21,Anne Arundel,5,5,5,...,10.0,NaN,I like the pre and post test \nI also like tha...,<NA>,150.0,False,0,1,1,False
4,5,2025-10-21 14:09:38,2025-10-21 14:10:49,English (United States),School staff member;,2025-10-21,Anne Arundel,5,4,5,...,9.0,NaN,The range of services offered through 988,Allow for a longer training with more interact...,71.0,False,0,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1294,1295,2026-10-25 00:40:44,2026-10-25 00:41:10,English (United States),Student,2026-10-09,Anne Arundel,5,5,5,...,10.0,NaN,988 is for everyone,<NA>,26.0,True,0,0,0,False
1295,1296,2026-10-25 00:41:14,2026-10-25 00:41:36,English (United States),Student,2026-10-09,Anne Arundel,3,4,4,...,5.0,NaN,<NA>,<NA>,22.0,True,0,2,1,False
1296,1297,2026-10-25 00:41:38,2026-10-25 00:42:14,English (United States),Student,2026-10-09,Anne Arundel,1,1,1,...,5.0,NaN,You don't have to call for suicide thoughts,<NA>,36.0,True,0,0,0,True
1297,1298,2026-10-25 00:42:17,2026-10-25 00:42:45,English (United States),Student,2026-10-09,Anne Arundel,5,5,5,...,2.0,NaN,I have people to talk to,<NA>,28.0,True,1,0,0,False


In [14]:
df.describe()

,response_id,survey_start,survey_end,post_identify_crisis,post_describe_988,post_know_when_988,pre_identify_crisis,pre_describe_988,pre_know_when_988,presenter_engaged,presenter_knowledgeable,training_organized,recommend_score_0_10,likely_to_call_988_0_10,duration_sec,gain_identify_crisis,gain_describe_988,gain_know_when_988
count,1299.00000,1299,1299,1299.000000,1299.000000,1299.000000,1299.000000,1299.000000,1299.000000,1125.000000,1123.000000,1118.000000,1266.000000,293.000000,1299.000000,1299.000000,1299.000000,1299.000000
mean,650.00000,2026-07-11 02:45:15.893764,2026-07-11 02:47:16.832178,4.250192,4.342571,4.442648,3.791378,3.229407,3.437259,4.327111,4.450579,4.469589,7.586888,7.658703,120.938414,0.458814,1.113164,1.005389
min,1.00000,2025-10-21 14:08:00,2025-10-21 14:09:59,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,10.000000,-4.000000,-4.000000,-4.000000
25%,325.50000,2026-05-04 15:47:58,2026-05-04 15:50:32.500000,4.000000,4.000000,4.000000,3.000000,2.000000,3.000000,4.000000,4.000000,4.000000,6.000000,6.000000,33.000000,0.000000,0.000000,0.000000
50%,650.00000,2026-07-26 17:31:43,2026-07-26 17:33:26,4.000000,5.000000,5.000000,4.000000,3.000000,4.000000,5.000000,5.000000,5.000000,8.000000,8.000000,47.000000,0.000000,1.000000,1.000000
75%,974.50000,2026-10-06 17:43:10.500000,2026-10-06 17:43:36.500000,5.000000,5.000000,5.000000,4.000000,4.000000,4.000000,5.000000,5.000000,5.000000,10.000000,10.000000,81.000000,1.000000,2.000000,2.000000
max,1299.00000,2026-10-25 00:42:48,2026-10-25 00:43:25,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,10.000000,10.000000,8295.000000,4.000000,4.000000,4.000000
std,375.13331,NaN,NaN,0.858963,0.873579,0.868236,0.903879,1.183358,1.243670,0.851847,0.842291,0.814190,2.273919,2.419860,425.971930,0.863819,1.265996,1.295649
